# Exploratory Data Analysis

Use this notebook to inspect the Hillstrom dataset, understand feature distributions, and identify preprocessing needs.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
data_dir = Path("..") / "data" / "raw"
dataset_path = data_dir / "hillstrom_email.csv"

if not dataset_path.exists():
    raise FileNotFoundError(
        "Could not find data/raw/hillstrom_email.csv."
    )

df = pd.read_csv(dataset_path)
print(f"Loaded {dataset_path.name} with shape {df.shape}")
df


Loaded hillstrom_email.csv with shape (64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0


In [3]:
# Data Types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  object 
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  object 
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  object 
 8   segment          64000 non-null  object 
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), object(4)
memory usage: 5.9+ MB


In [4]:
# checking for missing values
missing_summary = (
    df.isna()
    .sum()
    .to_frame("missing_count")
)
missing_summary

,missing_count
recency,0
history_segment,0
history,0
mens,0
womens,0
zip_code,0
newbie,0
channel,0
segment,0
visit,0


In [5]:
# checking for duplicates
duplicate_count = df.duplicated().sum()

print(f"Completely duplicated rows: {duplicate_count:,}")

Completely duplicated rows: 6,562


In [6]:
if duplicate_count > 0:
    display(
        df[df.duplicated(keep=False)]
        .sort_values(df.columns.tolist())
        .head(20)
    )

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
18473,1,1) $0 - $100,29.99,0,1,Rural,0,Phone,Mens E-Mail,0,0,0.0
38460,1,1) $0 - $100,29.99,0,1,Rural,0,Phone,Mens E-Mail,0,0,0.0
42755,1,1) $0 - $100,29.99,0,1,Rural,0,Phone,Mens E-Mail,0,0,0.0
47127,1,1) $0 - $100,29.99,0,1,Rural,0,Phone,Mens E-Mail,0,0,0.0
51350,1,1) $0 - $100,29.99,0,1,Rural,0,Phone,Mens E-Mail,0,0,0.0
3748,1,1) $0 - $100,29.99,0,1,Rural,0,Phone,Mens E-Mail,1,0,0.0
30423,1,1) $0 - $100,29.99,0,1,Rural,0,Phone,Mens E-Mail,1,0,0.0
50187,1,1) $0 - $100,29.99,0,1,Rural,0,Phone,Mens E-Mail,1,0,0.0
13562,1,1) $0 - $100,29.99,0,1,Rural,0,Phone,Womens E-Mail,0,0,0.0
21256,1,1) $0 - $100,29.99,0,1,Rural,0,Phone,Womens E-Mail,0,0,0.0


## Treatment Counts

In [8]:
df["treatment_binary"] = (df["segment"] != "No E-Mail").astype(int)
df["treatment_label"] = df["treatment_binary"].map({0: "No E-Mail", 1: "Any E-Mail"})

segment_counts = (
    df["segment"]
    .value_counts()
    .rename_axis("segment")
    .reset_index(name="customers")
)
segment_counts["share"] = segment_counts["customers"] / len(df)

binary_counts = (
    df["treatment_label"]
    .value_counts()
    .rename_axis("treatment_label")
    .reset_index(name="customers")
)
binary_counts["share"] = binary_counts["customers"] / len(df)

display(segment_counts)
display(binary_counts)

,segment,customers,share
0,Womens E-Mail,21387,0.334172
1,Mens E-Mail,21307,0.332922
2,No E-Mail,21306,0.332906


,treatment_label,customers,share
0,Any E-Mail,42694,0.667094
1,No E-Mail,21306,0.332906


The three original groups are almost perfectly balanced at about one-third of the dataset each. Womens E-Mail has 21,387 customers, Mens E-Mail has 21,307, and No E-Mail has 21,306.

## Outcome Summary

In [10]:
outcome_summary = pd.DataFrame(
    {
        "mean": df[["visit", "conversion", "spend"]].mean(),
        "std": df[["visit", "conversion", "spend"]].std(),
        "nonzero_rate": (df[["visit", "conversion", "spend"]] > 0).mean(),
    }
)

outcome_summary

,mean,std,nonzero_rate
visit,0.146781,0.353890,0.146781
conversion,0.009031,0.094604,0.009031
spend,1.050908,15.036448,0.009031


In [11]:
segment_outcomes = (
    df.groupby("segment")[["visit", "conversion", "spend"]]
    .agg(["mean", "count"])
)

segment_outcomes

visit        conversion            spend       
                   mean  count       mean  count      mean  count
segment                                                          
Mens E-Mail    0.182757  21307   0.012531  21307  1.422617  21307
No E-Mail      0.106167  21306   0.005726  21306  0.652789  21306
Womens E-Mail  0.151400  21387   0.008837  21387  1.077202  21387

The men’s e-mail group has the highest observed response across all three outcomes.
visit rates are about: 18.28% for Mens E-Mail, 15.14% for Womens E-Mail, 10.62% for No E-Mail.

Conversion is much sparser: 1.25% for Mens E-Mail, 0.88% for Womens E-Mail, 0.57% for No E-Mail.

Average spend also follows the same pattern: about 1.423 for Mens E-Mail, 1.077 for Womens E-Mail, 0.653 for No E-Mail

This supports using visit as the first modeling target because it is less sparse than conversion and easier to stabilize than spend.

## Feature Inventory

In [12]:
numeric_cols = ["recency", "history"]
binary_cols = ["mens", "womens", "newbie"]
categorical_cols = ["history_segment", "zip_code", "channel", "segment"]

display(df[numeric_cols].describe().T)

categorical_summary = pd.DataFrame(
    {
        "n_unique": df[categorical_cols + binary_cols].nunique(),
        "sample_values": [
            sorted(df[col].astype(str).unique())[:5]
            for col in categorical_cols + binary_cols
        ],
    },
    index=categorical_cols + binary_cols,
)

categorical_summary

,count,mean,std,min,25%,50%,75%,max
recency,64000.0,5.763734,3.507592,1.00,2.00,6.00,9.0000,12.00
history,64000.0,242.085656,256.158608,29.99,64.66,158.11,325.6575,3345.93


,n_unique,sample_values
history_segment,7,"[1) $0 - $100, 2) $100 - $200, 3) $200 - $350,..."
zip_code,3,"[Rural, Surburban, Urban]"
channel,3,"[Multichannel, Phone, Web]"
segment,3,"[Mens E-Mail, No E-Mail, Womens E-Mail]"
mens,2,"[0, 1]"
womens,2,"[0, 1]"
newbie,2,"[0, 1]"


The dataset mixes low-cardinality categorical features with a small number of numeric features.

## Covariate Balance Under Binary Treatment

In [15]:
balance_numeric = (
    df.groupby("treatment_label")[numeric_cols + binary_cols]
    .mean()
    .T
)

balance_numeric["diff_any_minus_none"] = (
    balance_numeric["Any E-Mail"] - balance_numeric["No E-Mail"]
)

balance_numeric

treatment_label,Any E-Mail,No E-Mail,diff_any_minus_none
recency,5.770741,5.749695,0.021046
history,242.686002,240.882653,1.803349
mens,0.549937,0.553224,-0.003288
womens,0.550757,0.547639,0.003117
newbie,0.502389,0.501971,0.000418


In [16]:
for col in ["zip_code", "channel", "history_segment"]:
    print(f"\n{col} distribution by treatment")
    display(pd.crosstab(df[col], df["treatment_label"], normalize="columns"))


zip_code distribution by treatment


treatment_label,Any E-Mail,No E-Mail
zip_code,,
Rural,0.150466,0.147329
Surburban,0.448564,0.451751
Urban,0.400970,0.400920



channel distribution by treatment


treatment_label,Any E-Mail,No E-Mail
channel,,
Multichannel,0.120766,0.122313
Phone,0.437860,0.437764
Web,0.441373,0.439923



history_segment distribution by treatment


treatment_label,Any E-Mail,No E-Mail
history_segment,,
1) $0 - $100,0.359723,0.357270
2) $100 - $200,0.220593,0.226978
3) $200 - $350,0.193118,0.189806
4) $350 - $500,0.100365,0.099690
5) $500 - $750,0.076334,0.077537
"6) $750 - $1,000",0.028974,0.029194
"7) $1,000 +",0.020893,0.019525


In [17]:
for col in ["zip_code", "channel", "history_segment"]:
    print(f"\n{col} distribution by treatment")
    display(pd.crosstab(df[col], df["treatment_label"], normalize="columns"))


zip_code distribution by treatment


treatment_label,Any E-Mail,No E-Mail
zip_code,,
Rural,0.150466,0.147329
Surburban,0.448564,0.451751
Urban,0.400970,0.400920



channel distribution by treatment


treatment_label,Any E-Mail,No E-Mail
channel,,
Multichannel,0.120766,0.122313
Phone,0.437860,0.437764
Web,0.441373,0.439923



history_segment distribution by treatment


treatment_label,Any E-Mail,No E-Mail
history_segment,,
1) $0 - $100,0.359723,0.357270
2) $100 - $200,0.220593,0.226978
3) $200 - $350,0.193118,0.189806
4) $350 - $500,0.100365,0.099690
5) $500 - $750,0.076334,0.077537
"6) $750 - $1,000",0.028974,0.029194
"7) $1,000 +",0.020893,0.019525


The tables show only small differences, which supports the current binary-treatment setup. Here, we confirmed that the experiment was randomized effectively. Treated and control customers are similar.

## Simple Heterogeneity Proxies

In [18]:
def subgroup_uplift(frame: pd.DataFrame, feature: str) -> pd.DataFrame:
    summary = (
        frame.groupby([feature, "treatment_label"])["visit"]
        .agg(["mean", "count"])
        .reset_index()
        .pivot(index=feature, columns="treatment_label", values=["mean", "count"])
    )
    summary.columns = [f"{metric}_{group}" for metric, group in summary.columns]
    summary = summary.reset_index()
    summary["uplift_proxy_visit"] = (
        summary["mean_Any E-Mail"] - summary["mean_No E-Mail"]
    )
    return summary.sort_values("uplift_proxy_visit", ascending=False)

In [19]:
for feature in ["channel", "zip_code", "history_segment", "mens", "womens", "newbie"]:
    print(f"\nSubgroup uplift proxy by {feature}")
    display(subgroup_uplift(df, feature))


Subgroup uplift proxy by channel


,channel,mean_Any E-Mail,mean_No E-Mail,count_Any E-Mail,count_No E-Mail,uplift_proxy_visit
0,Multichannel,0.193561,0.128550,5156.0,2606.0,0.065011
2,Web,0.179580,0.118852,18844.0,9373.0,0.060728
1,Phone,0.147106,0.087166,18694.0,9327.0,0.059940



Subgroup uplift proxy by zip_code


,zip_code,mean_Any E-Mail,mean_No E-Mail,count_Any E-Mail,count_No E-Mail,uplift_proxy_visit
2,Urban,0.160114,0.096816,17119.0,8542.0,0.063299
1,Surburban,0.160514,0.099013,19151.0,9625.0,0.061501
0,Rural,0.205012,0.153552,6424.0,3139.0,0.051460



Subgroup uplift proxy by history_segment


...

,mens,mean_Any E-Mail,mean_No E-Mail,count_Any E-Mail,count_No E-Mail,uplift_proxy_visit
0,0,0.168098,0.095808,19215.0,9519.0,0.072289
1,1,0.166191,0.114533,23479.0,11787.0,0.051658



Subgroup uplift proxy by womens


,womens,mean_Any E-Mail,mean_No E-Mail,count_Any E-Mail,count_No E-Mail,uplift_proxy_visit
1,1,0.189249,0.111416,23514.0,11668.0,0.077833
0,0,0.139833,0.099813,19180.0,9638.0,0.040020



Subgroup uplift proxy by newbie


,newbie,mean_Any E-Mail,mean_No E-Mail,count_Any E-Mail,count_No E-Mail,uplift_proxy_visit
1,1,0.141638,0.078822,21449.0,10695.0,0.062816
0,0,0.192704,0.133729,21245.0,10611.0,0.058975


There are many noticeable differences.